# Lightning AI Runner: Swin SSL + 17-class Segmentation (with augmentation)

Lightning-AI version of `colab_run_pipeline_221.ipynb`. Runs the **unchanged**
training pipeline (`swin_ssl_pretrain_221.py` -> `swin_training_pipeline_221.py`)
on a Lightning Studio with GPU.

**How this differs from the Colab version**
- Uses **rclone** (not `google.colab.drive`) to reach Google Drive.
- Data is synced down to the Studio's **local disk** (`/teamspace/studios/this_studio/...`)
  for fast I/O during training, then outputs are synced back up to Drive.
- Code is pulled from GitHub over SSH (no Drive-hosted clone).

**Assumptions (documented per request — change the config cell if any are wrong):**
- The repo is cloned at `~/Payne_Lab_Carbon_Thin_Segmentation` and this Studio's SSH key
  is registered on GitHub (so `git pull` needs no token).
- An rclone remote named **`gdrive`** points at the Google Drive whose root is `My Drive`.
- The labeled dataset lives at `My Drive/Petrographic images_ML work/labelled images_PS/ALL_LABELS`
  with `img/` and `masks_machine/`. (The task spec wrote `labelled_images PS`; the known-good
  folder is `labelled images_PS` — adjust `LABELLED_DRIVE` below if your Drive differs.)
- SSL pretraining uses the three unlabeled folders the pipeline is configured to scan:
  `cretaceous thin sections`, `Permian-Triassic`, `TJ photomicrographs`. The spec's
  "Fine-tuning dataset" line is treated as a *section header*, not a 4th pretraining folder,
  because `swin_ssl_pretrain_221.py` only scans those three (keeping the pipeline unchanged).
- "17-class" = the model's 18 logits (id 0 = background + 17 labeled classes 1..17). The
  pipeline is left at NUM_CLASSES=18; `--ignore_artifacts` excludes scale bar (11) and
  watermark (17) from the loss. "With augmentation" = the finetune trains on the combined `augmented_and_og_labels`
  dataset (the original labeled images + their pre-generated augmentations). CAUTION:
  K-fold CV on this combined set can leak augmentations of a validation image into
  training, which inflates the CV mIoU -- compare to the no-aug run with care.

---

### One-time Studio setup (run in the Lightning **terminal**, not this notebook)
```bash
# 1. Install rclone
curl https://rclone.org/install.sh | sudo bash

# 2. Configure a Google Drive remote named exactly `gdrive`
rclone config
#   n -> new remote ; name: gdrive ; storage: drive
#   client_id / client_secret: blank ; scope: 1 (full access)
#   service_account_file: blank ; advanced: n
#   auto config: n   <-- IMPORTANT (no browser on the Studio); open the URL on your
#                        laptop and paste the auth code back
#   team drive: n ; y ; q

# 3. Clone the repo (if not already) and verify both
cd ~ && git clone git@github.com:racyun/Payne_Lab_Carbon_Thin_Segmentation.git
rclone lsd gdrive:
```
**Resumable:** rclone syncs and the SSL `--resume` flag are idempotent — if the Studio
disconnects, re-run from the data-sync cell onward.


## 0. Pull latest code from GitHub

In [ ]:
import os
REPO_ROOT = os.path.expanduser('~/Payne_Lab_Carbon_Thin_Segmentation')
os.chdir(REPO_ROOT)
# This Studio's SSH key is registered on GitHub, so no token prompt is needed.
!git pull origin main
print('Repo:', REPO_ROOT)

## 1. Check runtime (GPU)

In [ ]:
import torch
USE_GPU = torch.cuda.is_available()
if USE_GPU:
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected. Open a GPU Studio (Settings -> Compute) before training —')
    print('SSL/finetune on CPU is impractical at these sizes.')

## 2. Install Python dependencies

Lightning Studios ship torch/torchvision; we add the pipeline's extra deps.

In [ ]:
!pip -q install --upgrade transformers tqdm wandb

## 3. (Optional) Weights & Biases login

Skip this cell to disable W&B logging (drop the `--wandb_*` flags below too).

In [ ]:
import os
os.environ.setdefault('WANDB_PROJECT', 'payne-carbonate-segmentation')
import wandb
wandb.login()

## 4. Verify rclone is set up

If this errors, finish the one-time rclone setup at the top of the notebook.

In [ ]:
import subprocess
res = subprocess.run(['rclone', 'lsd', 'gdrive:'], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError('rclone not configured. Error:\n' + res.stderr)
print('rclone OK. Top-level Drive folders:')
print(res.stdout)

## 5. Config — Drive paths and Studio-local paths

In [ ]:
import os
from pathlib import Path

# --- Google Drive side (relative to the `gdrive:` remote root, i.e. 'My Drive') ---
DRIVE_ROOT = 'Petrographic images_ML work'
# Labeled segmentation dataset. NOTE: spec wrote 'labelled_images PS'; using the known-good
# 'labelled images_PS'. Change here if your Drive uses a different spelling.
LABELLED_DRIVE = DRIVE_ROOT + '/augmented_and_og_labels'
# Unlabeled folders for SSL (the three the pipeline is configured to scan).
UNLABELED_SUBFOLDERS = ['cretaceous thin sections', 'Permian-Triassic', 'TJ photomicrographs']

# --- Studio-local disk (fast I/O; space-free paths so no shell quoting is needed) ---
LOCAL_ROOT         = Path('/teamspace/studios/this_studio/petro_data')
UNLABELED_LOCAL    = LOCAL_ROOT / 'unlabeled'
LABELED_IMG_LOCAL  = LOCAL_ROOT / 'labeled' / 'img'
LABELED_MASK_LOCAL = LOCAL_ROOT / 'labeled' / 'masks_machine'
OUT_ROOT           = LOCAL_ROOT / 'model_outputs'
for d in (UNLABELED_LOCAL, LABELED_IMG_LOCAL, LABELED_MASK_LOCAL, OUT_ROOT):
    d.mkdir(parents=True, exist_ok=True)

# Where to sync outputs back to on Drive.
OUT_DRIVE = DRIVE_ROOT + '/model_outputs_lightning'

# Python + env handles for the !python training cells.
IMG_DIR  = str(LABELED_IMG_LOCAL)
MASK_DIR = str(LABELED_MASK_LOCAL)
os.environ['REPO_ROOT']       = REPO_ROOT
os.environ['UNLABELED_LOCAL'] = str(UNLABELED_LOCAL)
os.environ['IMG_DIR']         = IMG_DIR
os.environ['MASK_DIR']        = MASK_DIR
os.environ['OUT_ROOT']        = str(OUT_ROOT)

print('Drive root  : gdrive:' + DRIVE_ROOT)
print('Labeled     : gdrive:' + LABELLED_DRIVE)
print('Local root  :', LOCAL_ROOT)
print('Output (loc):', OUT_ROOT)

## 6. Sync datasets from Drive -> Studio disk (rclone)

`rclone copy` only transfers missing/changed files, so re-running is cheap and resumable.
The unlabeled subfolders are pulled preserving their names, so `--gdrive_root UNLABELED_LOCAL`
finds exactly the three folders the SSL script scans.

In [ ]:
import subprocess, os
from pathlib import Path

def rclone_copy(remote_rel, local_path):
    local_path = Path(local_path); local_path.mkdir(parents=True, exist_ok=True)
    remote = 'gdrive:' + remote_rel
    print('  pulling', remote, '->', local_path)
    subprocess.run(['rclone', 'copy', remote, str(local_path),
                    '--progress', '--transfers=8', '--checkers=16'], check=True)

# Unlabeled (SSL pretraining)
for sub in UNLABELED_SUBFOLDERS:
    rclone_copy(DRIVE_ROOT + '/' + sub, UNLABELED_LOCAL / sub)

# Labeled (segmentation finetune) — pair img/<stem> with masks_machine/<stem>
rclone_copy(LABELLED_DRIVE + '/img', LABELED_IMG_LOCAL)
rclone_copy(LABELLED_DRIVE + '/masks_machine', LABELED_MASK_LOCAL)

n_unl = sum(len(fs) for _, _, fs in os.walk(UNLABELED_LOCAL))
print('\nUnlabeled files (SSL):', n_unl)
print('Labeled images       :', len(list(LABELED_IMG_LOCAL.glob('*'))))
print('Labeled masks        :', len(list(LABELED_MASK_LOCAL.glob('*'))))

## 7. Smoke tests (recommended first)

In [ ]:
import os
os.chdir(REPO_ROOT)

# SSL smoke test: 1 epoch, 2 steps
!python -u code/model_training_pipeline/swin_ssl_pretrain_221.py \
  --gdrive_root "$UNLABELED_LOCAL" \
  --epochs 1 --batch_size 2 --num_workers 2 --max_steps_per_epoch 2 \
  --output_dir "$OUT_ROOT/ssl_smoke" --amp

# Segmentation dataloader smoke test (no training)
!python -u code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" --mask_dir "$MASK_DIR" --no_train

## 8. Full SSL pretraining (stage 1)

150 epochs of masked-image pretraining on the local unlabeled copy. Launched in the
background with output to a log file (tail it in the next cell). `--resume` auto-engages
if a previous `ssl_swinv2_last.pth` exists. Local paths have no spaces, so the command is
built without shell quoting.

In [ ]:
import os
from pathlib import Path
os.chdir(REPO_ROOT)

ssl_out = OUT_ROOT / 'ssl_full'
ssl_out.mkdir(parents=True, exist_ok=True)
ssl_log = ssl_out / 'ssl_run.log'
os.environ['WANDB_DIR'] = str(OUT_ROOT / 'wandb'); os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)
os.environ['SSL_LOG'] = str(ssl_log)

resume_path = ssl_out / 'ssl_swinv2_last.pth'
resume_arg = f'--resume {resume_path}' if resume_path.exists() else ''
print('Resume mode:', 'ON' if resume_arg else 'OFF (starting fresh)')
print('Log file   :', ssl_log)

cmd = (
    'python -u code/model_training_pipeline/swin_ssl_pretrain_221.py '
    f'--gdrive_root {UNLABELED_LOCAL} '
    '--epochs 150 --batch_size 8 --num_workers 4 --crop 512 '
    '--mask_patch 16 --mask_ratio 0.70 '
    '--save_recon_every 1 --save_last_every 5 --num_recon_samples 2 '
    f'--output_dir {ssl_out} --amp '
    '--wandb_project payne-carbonate-segmentation --wandb_run_name ssl_lightning_mask070_e150 '
    f'{resume_arg} '
    f'> {ssl_log} 2>&1 &'
)
print('\nLaunching:\n', cmd, sep='')
get_ipython().system(cmd)
print('\nSSL launched in background. Tail the log with the next cell.')

## 8b. Tail the live SSL log (stop button does NOT kill training)

In [ ]:
!tail -f "$SSL_LOG"

## 9. Full segmentation finetune (stage 2)

**Focal** CE, artifacts (scale bar + watermark) ignored, cosine LR with 5-epoch warmup,
50 epochs, SSL-initialized backbone. Defaults to **3-fold stratified CV** (writes per-fold
checkpoints under `fold_<i>/best_upernet_swinv2.pth` plus an aggregate `cv_summary.json`).
Pass `--n_folds 1` for a single train/val split instead. (No `--auto_class_weights`: with the
ignored artifact classes it produces degenerate near-zero weights and the loss flatlines;
focal loss handles class imbalance.)

In [ ]:
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --epochs 50 \
  --batch_size 2 \
  --crop 512 \
  --lr 3e-4 \
  --warmup_epochs 5 \
  --scheduler cosine \
  --ignore_artifacts \
  --group_by_stem \
  --loss_type focal \
  --focal_gamma 2.0 \
  --backbone_checkpoint "$OUT_ROOT/ssl_full/ssl_swinv2_best.pth" \
  --output_dir "$OUT_ROOT/seg_with_aug_3fold" \
  --wandb_project payne-carbonate-segmentation \
  --wandb_run_name seg_with_aug_17class_focal_3fold

## 10. Sync outputs back to Google Drive (rclone)

Uploads checkpoints, logs, and previews; only new/changed files transfer.

In [ ]:
import subprocess
subprocess.run(['rclone', 'copy', str(OUT_ROOT), 'gdrive:' + OUT_DRIVE,
                '--progress', '--transfers=8', '--checkers=16'], check=True)
print('Outputs synced to gdrive:' + OUT_DRIVE)

## 11. Confusion matrix on a chosen fold's validation split

Loads a chosen fold's finetuned checkpoint (`fold_<FOLD>/best_upernet_swinv2.pth`) and rebuilds
the SAME stratified validation split that fold trained against, runs inference, and saves
row- and column-normalized confusion matrices plus a per-class recall/precision/IoU table.
Set `FOLD`/`N_FOLDS`/`CV_STRATEGY` to match the finetune run. Artifacts are remapped to
ignore_index to match `--ignore_artifacts`.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from torchvision.transforms.v2 import CenterCrop, Compose

PIPE_DIR = Path(REPO_ROOT) / 'code' / 'model_training_pipeline'
if str(PIPE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPE_DIR))
from swin_training_pipeline_221 import (
    CLASS_NAMES, NUM_CLASSES, IGNORE_INDEX, BACKBONE_ID, ARTIFACT_CLASS_IDS,
    CarbonateSegmentationDataset, confusion_matrix as cm_fn, get_model_semantic_segmentation,
    build_class_presence_matrix, stratified_kfold_indices, kfold_train_val_indices,
    group_members_by_stem, stratified_grouped_kfold_indices, grouped_kfold_indices,
)

# --- Config: MUST match the finetune run (cell 22) ---
SEG_OUT       = OUT_ROOT / 'seg_with_aug_3fold'
FOLD          = 0                  # which fold to evaluate (0 .. N_FOLDS-1)
N_FOLDS       = 3                  # must equal --n_folds used in training
CV_STRATEGY   = 'stratified'      # must equal --cv_strategy
GROUP_BY_STEM = True              # must match --group_by_stem (families kept in one fold)
GROUP_PATTERN = r'_aug\d+$'
SEED          = 1337
CROP          = 512
IGNORE_IDS    = ARTIFACT_CLASS_IDS  # match --ignore_artifacts
CKPT          = SEG_OUT / f'fold_{FOLD}' / 'best_upernet_swinv2.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Rebuild the exact validation split fold FOLD used (grouped if GROUP_BY_STEM) ---
probe = CarbonateSegmentationDataset(
    root='.', img_dir=IMG_DIR, mask_dir=MASK_DIR,
    transforms=None, normalize=True, strict=True, ignore_class_ids=IGNORE_IDS,
)
n = len(probe)
presence = (build_class_presence_matrix(probe.pairs, NUM_CLASSES, IGNORE_INDEX, IGNORE_IDS)
            if CV_STRATEGY == 'stratified' else None)
members = group_members_by_stem(probe.pairs, GROUP_PATTERN) if GROUP_BY_STEM else None
if GROUP_BY_STEM and CV_STRATEGY == 'stratified':
    splits = stratified_grouped_kfold_indices(presence, members, N_FOLDS, SEED)
elif GROUP_BY_STEM:
    splits = grouped_kfold_indices(members, N_FOLDS, SEED)
elif CV_STRATEGY == 'stratified':
    splits = stratified_kfold_indices(presence, N_FOLDS, SEED)
else:
    splits = kfold_train_val_indices(n, N_FOLDS, SEED)
val_idx = list(np.asarray(splits[FOLD][1]).tolist())

val_tf = Compose([CenterCrop((CROP, CROP))])
val_full = CarbonateSegmentationDataset(
    root='.', img_dir=IMG_DIR, mask_dir=MASK_DIR,
    transforms=val_tf, normalize=True, strict=False, print_pair_count=False,
    ignore_class_ids=IGNORE_IDS,
)
val_ds = Subset(val_full, val_idx)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())
print(f'Fold {FOLD}/{N_FOLDS} | val samples: {len(val_ds)} | grouped={GROUP_BY_STEM}')
print(f'Checkpoint: {CKPT}')

# --- Load that fold's finetuned model ---
model = get_model_semantic_segmentation(NUM_CLASSES, IGNORE_INDEX, BACKBONE_ID).to(device)
ckpt = torch.load(str(CKPT), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded {CKPT}')

# --- Accumulate the 18x18 confusion matrix over the fold's val set ---
cm_total = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.float64)
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(pixel_values=imgs).logits
        logits = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        preds  = logits.argmax(dim=1)
        cm_total += cm_fn(preds, labels, NUM_CLASSES, IGNORE_INDEX).cpu().double()

cm_np = cm_total.numpy()
out_dir = SEG_OUT / f'fold_{FOLD}'
np.save(out_dir / 'cm_raw.npy', cm_np)
print('Saved', out_dir / 'cm_raw.npy', 'shape', cm_np.shape)

row_sums = cm_np.sum(axis=1, keepdims=True); row_sums[row_sums == 0] = 1
col_sums = cm_np.sum(axis=0, keepdims=True); col_sums[col_sums == 0] = 1
cm_row = cm_np / row_sums   # diagonal = recall
cm_col = cm_np / col_sums   # diagonal = precision

def plot_cm(mat, title, out_path):
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(mat, vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth'); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = mat[i, j]
            if v > 0.005:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=7, color='white' if v > 0.5 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)
    print('Saved', out_path)

plot_cm(cm_row, f'Fold {FOLD} confusion matrix (row-normalized; diag = recall)',
        out_dir / 'cm_row_normalized.png')
plot_cm(cm_col, f'Fold {FOLD} confusion matrix (col-normalized; diag = precision)',
        out_dir / 'cm_col_normalized.png')

diag = np.diag(cm_np)
recall    = diag / np.maximum(cm_np.sum(axis=1), 1)
precision = diag / np.maximum(cm_np.sum(axis=0), 1)
union     = cm_np.sum(axis=1) + cm_np.sum(axis=0) - diag
iou       = np.where(union > 0, diag / np.maximum(union, 1), np.nan)

print(f'\nFold {FOLD} per-class metrics:')
print(f"{'class':<20}{'recall':>10}{'precision':>12}{'IoU':>10}")
for i, name in enumerate(CLASS_NAMES):
    iou_s = f'{iou[i]:.3f}' if np.isfinite(iou[i]) else '  nan'
    print(f'{name:<20}{recall[i]:>10.3f}{precision[i]:>12.3f}{iou_s:>10}')